In [ ]:
!pip install -q sentence-transformers faiss-cpu datasets google-generativeai

In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset

import os

from sentence_transformers import SentenceTransformer

import faiss

import google.generativeai as genai

In [ ]:
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

print(dataset)

In [ ]:
print(dataset[0])

In [ ]:
df = pd.DataFrame(dataset)

df.head()

In [ ]:
print(df.shape)

In [ ]:
df.columns

In [ ]:
df[['title', 'abstract']].head()

In [ ]:
df[['title','abstract']].isnull().sum()

In [ ]:
print("Duplicate Papers:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

print("New Shape:", df.shape)

In [ ]:
df["paper_text"] = df["title"] + ". " + df["abstract"]

In [ ]:
df[["title", "paper_text"]].head()

In [ ]:
print(df["paper_text"].iloc[0])

In [ ]:
df["paper_length"] = df["paper_text"].apply(len)

df["paper_length"].describe()

In [ ]:
df.to_csv("cleaned_research_papers.csv", index=False)

print("Cleaned dataset saved successfully!")

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading Sentence Transformer Model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully!")

In [ ]:
print("Embedding Dimension:", model.get_sentence_embedding_dimension())

In [ ]:
sample_embedding = model.encode(df["paper_text"].iloc[0])

print(sample_embedding)

In [ ]:
print(sample_embedding.shape)

In [ ]:
print("First 20 values:\n")

print(sample_embedding[:20])

In [ ]:


df = df.head(10000).copy()

print("Number of papers:", len(df))

In [ ]:
import numpy as np

In [ ]:
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings...")

    embeddings = np.load("paper_embeddings.npy")

else:
    print("Generating embeddings...")

    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save("paper_embeddings.npy", embeddings)

    print("Embeddings saved successfully!")

In [ ]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index...")

    index = faiss.read_index("paper_faiss.index")

else:
    print("Creating new FAISS index...")

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    # Create FAISS index
    index = faiss.IndexFlatIP(384)

    # Add embeddings
    index.add(embeddings)

    # Save index
    faiss.write_index(index, "paper_faiss.index")

    print("FAISS index saved successfully!")

In [ ]:
print("Total vectors stored:", index.ntotal)

In [ ]:
def search_papers(query, top_k=5):

    # Convert query to embedding
    query_embedding = model.encode([query])

    # Normalize query
    faiss.normalize_L2(query_embedding)

    # Search
    distances, indices = index.search(query_embedding, top_k)

    print("=" * 80)
    print("Query:", query)
    print("=" * 80)

    for i, idx in enumerate(indices[0], 1):

        print(f"\nResult {i}")
        print("-" * 80)

        print("Title:")
        print(df.iloc[idx]["title"])

        print("\nAbstract:")
        print(df.iloc[idx]["abstract"][:500] + "...")

        print("\nSimilarity Score:", round(float(distances[0][i-1]), 4))

In [ ]:
search_papers("Large Language Models")

In [ ]:
search_papers("Deep Learning")

In [ ]:
search_papers("Computer Vision")

In [ ]:
search_papers("Neural Networks")

In [ ]:
search_papers("Medical AI")

In [ ]:
search_papers("Reinforcement Learning")

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai

In [ ]:


from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini Connected Successfully!")

In [ ]:
paper = df.iloc[0]["paper_text"]

prompt = f"""
You are an AI research assistant.

Analyze the following research paper and provide the response in this format:

1. Paper Title
2. Short Summary (3-4 lines)
3. Main Objective
4. Key Contributions (Bullet Points)
5. Methodology Used
6. Important Findings
7. Limitations
8. Future Scope
9. Explain Like I'm a Beginner

Research Paper:

{paper}
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

In [ ]:
def search_papers(query, top_k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, top_k)

    results = []

    print("=" * 80)
    print(f"Query : {query}")
    print("=" * 80)

    for rank, idx in enumerate(indices[0], start=1):

        title = df.iloc[idx]["title"]

        score = float(distances[0][rank-1])

        results.append(idx)

        print(f"\n{rank}. {title}")
        print(f"Similarity Score : {score:.4f}")

    return results

In [ ]:
def summarize_search_result(query, result_number=1):

    results = search_papers(query)

    paper_index = results[result_number-1]

    paper = df.iloc[paper_index]["paper_text"]

    prompt = f"""
You are an AI Research Assistant.

Analyze the following research paper.

Provide:

1. Summary

2. Key Contributions

3. Methodology

4. Important Findings

5. Limitations

6. Future Scope

7. Explain Like I'm a Beginner

Research Paper:

{paper}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    print("\n")
    print("="*80)
    print("AI SUMMARY")
    print("="*80)

    print(response.text)

In [ ]:
summarize_search_result(
    "Large Language Models",
    1
)